In [1]:
import random

import wandb
run = wandb.init(
    entity="id25s027-iit-madras",
    project="da6401-assignment1-puneet-id25s027",
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset
    run.log({"acc": acc, "loss": loss})
run.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\brpun\_netrc.
wandb: Currently logged in as: id25s027 (id25s027-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


acc,▁▆▇▇██▇▇
loss,█▆▄▂▃▂▁▁
acc,0.82132
loss,0.13287


In [2]:
import os
import ast
import wandb
import numpy as np
from typing import List
import matplotlib.pyplot as plt
from keras.utils import to_categorical
from utils.data_loader import load_dataset
from ann.neural_network import NeuralNetwork
from sklearn.model_selection import train_test_split


In [3]:
class Config:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)


In [4]:
PROJECT_NAME = "da6401-assignment1-puneet-id25s027"

In [5]:
dataset = "mnist"

(train_images, train_labels), (test_images, test_labels) = load_dataset(dataset)

In [6]:
run = wandb.init(
            project = PROJECT_NAME
        )

wandb.run.name = "Data-Exploration-and-Class-Distribution"

In [7]:
wandb_images = []

for i in range(10):
    sample_images = train_images[train_labels == i][:5]
    for im in sample_images:
        wandb_images.append(wandb.Image(im, caption=i))

run.log({"mnist_images": wandb_images})
# i = 0

run.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [8]:
from PIL import Image

os.makedirs("mnist_samples", exist_ok=True)

run = wandb.init(
            project = PROJECT_NAME
        )

wandb.run.name = "Data-Exploration-and-Class-Distribution"

wandb_images_table = wandb.Table(columns=["id", "image", "label"])

wandb_images = []

j = 0
for i in range(10):
    sample_images = train_images[train_labels == i][:5]
    for im in sample_images:
        image = wandb.Image(im)
        wandb_images_table.add_data(j+1, image, i)
        j = j + 1


run.log({"mnist_images_table": wandb_images_table})

run.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [9]:
train_images, test_images = train_images / 255.0, test_images / 255.0

train_images = train_images.reshape((-1, 28 * 28))
test_images = test_images.reshape((-1, 28 * 28))

output_size = len(np.unique(train_labels).tolist())

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

x_train, x_val, y_train, y_val = train_test_split(train_images, train_labels, test_size = 0.1, random_state = 42)

print(train_images[0].shape[0])

784


In [10]:
def model_run(config, output_activation):
    model = NeuralNetwork(
            cli_args=config,
            input_size = train_images[0].shape[0],
            output_size = 10,
            output_activation = output_activation
        )
    run = wandb.init(
            project = PROJECT_NAME
        )
    wandb.run.name = f"[The-Optimizer-Showdown]-lr={config.learning_rate}_bs={config.batch_size}_opt={config.optimizer}_act={config.activation}_loss={config.loss}_wd={config.weight_decay}_wi={config.weight_init}_sz={config.hidden_layers}_wandbid={wandb.run.id}"
        
    model.train(x_train, y_train, epochs=config.epochs, batch_size=config.batch_size, x_val=x_val, y_val=y_val, wandb=run)

    run.finish()

In [11]:
optimizers = ["sgd", "momentum", "nag", "rmsprop", 'adam', 'nadam']


config = {
    "learning_rate": 0.01,
    "activation": "relu",
    "epochs": 15,
    "hidden_layers": [128,128,128],
    "loss": "cross_entropy",
    "weight_init": "random",
    "weight_decay": 0.0,
    "batch_size": 64,
    "wandb_project": PROJECT_NAME
}

output_activation = "softmax"


for optimizer in optimizers:
    config["optimizer"] = optimizer
    
    cfg = Config(**config)

    model_run(cfg, output_activation=output_activation)

Epoch   1/15 : Train Loss = 0.4018, Train Accuracy = 89.4074074074074 | Val Loss = 0.4128, Val Accuracy = 88.9667
Epoch   6/15 : Train Loss = 0.0587, Train Accuracy = 98.18518518518519 | Val Loss = 0.0966, Val Accuracy = 97.1500
Epoch  11/15 : Train Loss = 0.0520, Train Accuracy = 98.44074074074074 | Val Loss = 0.1133, Val Accuracy = 97.0167


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch  15/15 : Train Loss = 0.0257, Train Accuracy = 99.17777777777778 | Val Loss = 0.1009, Val Accuracy = 97.5000


best_val_loss,▁
convergence_epoch,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
final_train_accuracy,▁
final_train_loss,▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,█▇█▆▆▅▅▄▅▃▃▄▂▁▂
train_accuracy,▁▅▅▇▇▇▇███▇█▇██
train_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
+3,...


Epoch   1/15 : Train Loss = 2.3185, Train Accuracy = 11.274074074074074 | Val Loss = 2.3203, Val Accuracy = 10.9000
Epoch   6/15 : Train Loss = 2.3211, Train Accuracy = 10.262962962962963 | Val Loss = 2.3212, Val Accuracy = 9.8167
Epoch  11/15 : Train Loss = 2.3172, Train Accuracy = 9.751851851851852 | Val Loss = 2.3202, Val Accuracy = 9.7500


...

Epoch   1/15 : Train Loss = 0.3998, Train Accuracy = 88.16111111111111 | Val Loss = 0.4108, Val Accuracy = 87.9000
Epoch   6/15 : Train Loss = 0.2144, Train Accuracy = 94.03148148148148 | Val Loss = 0.2634, Val Accuracy = 92.9667
Epoch  11/15 : Train Loss = 0.1845, Train Accuracy = 95.24259259259259 | Val Loss = 0.2520, Val Accuracy = 93.9667


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch  15/15 : Train Loss = 0.2130, Train Accuracy = 94.37962962962962 | Val Loss = 0.2700, Val Accuracy = 93.6167


best_val_loss,▁
convergence_epoch,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
final_train_accuracy,▁
final_train_loss,▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,█▆▇▅▅▄▅▄▃▅▃▃▃▁▄
train_accuracy,▂▃▁▆▇▇█▇▇▇█▇█▇▇
train_loss,█▆▇▃▃▂▁▂▂▃▁▂▁▄▂
+3,...


<Figure size 640x480 with 0 Axes>

In [12]:
objectives = ["cross_entropy", "mse"]

In [21]:
!python train.py --dataset mnist --epochs 20 --batch_size 64 --learning_rate 0.1 --optimizer adam --hidden_layers 32 64 64 32 --num_neurons 20 --activation sigmoid --loss cross_entropy --weight_init random --weight_decay 0.0 --wandb_project da6401-assignment1-puneet-id25s027 --question Loss-Function-Comparison

Epoch   1/20 : Train Loss = 2.3149, Train Accuracy = 9.812962962962963 | Val Loss = 2.3127, Val Accuracy = 10.4000
Epoch   6/20 : Train Loss = 2.3051, Train Accuracy = 11.274074074074074 | Val Loss = 2.3043, Val Accuracy = 10.9000
Epoch  11/20 : Train Loss = 2.3195, Train Accuracy = 9.885185185185186 | Val Loss = 2.3204, Val Accuracy = 9.6667
Epoch  16/20 : Train Loss = 2.3080, Train Accuracy = 9.751851851851852 | Val Loss = 2.3069, Val Accuracy = 9.7500
Epoch  20/20 : Train Loss = 2.3051, Train Accuracy = 11.274074074074074 | Val Loss = 2.3070, Val Accuracy = 10.9000


2026-03-07 15:29:17.695158: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 15:29:22.345802: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\brpun\_netrc.
wandb: Currently logged in as: id25s027 (id25s027-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 8lc5qgqb
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in c:\My Folder\projects\da6401_assignment_1_id25s027\src\wandb\run-2026030

In [22]:
!python train.py --dataset mnist --epochs 20 --batch_size 64 --learning_rate 0.1 --optimizer adam --hidden_layers 32 64 64 32 --num_neurons 20 --activation sigmoid --loss mse --weight_init random --weight_decay 0.0 --wandb_project da6401-assignment1-puneet-id25s027 --question Loss-Function-Comparison

Epoch   1/20 : Train Loss = 0.1000, Train Accuracy = 9.974074074074075 | Val Loss = 0.1000, Val Accuracy = 9.5333
Epoch   6/20 : Train Loss = 0.1000, Train Accuracy = 9.974074074074075 | Val Loss = 0.1000, Val Accuracy = 9.5333
Epoch  11/20 : Train Loss = 0.1000, Train Accuracy = 10.429629629629629 | Val Loss = 0.1000, Val Accuracy = 10.5500
Epoch  16/20 : Train Loss = 0.1000, Train Accuracy = 10.429629629629629 | Val Loss = 0.1000, Val Accuracy = 10.5500
Epoch  20/20 : Train Loss = 0.1000, Train Accuracy = 9.812962962962963 | Val Loss = 0.1000, Val Accuracy = 10.4000


2026-03-07 16:03:39.924920: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 16:03:47.194346: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\brpun\_netrc.
wandb: Currently logged in as: id25s027 (id25s027-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run jzvmlzix
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in c:\My Folder\projects\da6401_assignment_1_id25s027\src\wandb\run-2026030

In [23]:
from inference import load_model, evaluate_model

In [29]:
model_path = "best_model.npy"
model = load_model(model_path=model_path)
evals = evaluate_model(model, test_images, test_labels)

In [31]:
# run = wandb.init(
#     project = PROJECT_NAME
# )

# wandb.run.name = "[Error-Analysis]"
# confusion_matrix = wandb.plot.confusion_matrix(
#     probs  = None,
#     y_true = evals["y_true"], 
#     preds  = evals["y_pred"],
#     class_names=[0,1,2,3,4,5,6,7,8,9])


# misclassified_images = wandb.Table(columns=["Index", "Image", "Actual", "Predicted"])

# for i, (y, y_) in enumerate(zip(evals["y_true"], evals["y_pred"])):
#     if y!=y_:
#         misclassified_images.add_data(i, wandb.Image(test_images[i]), y, y_)


# run.log({"confusion_matrix": confusion_matrix, "misclassified_images": misclassified_images})

# run.finish()

In [ ]:
import numpy as np
import wandb

run = wandb.init(project=PROJECT_NAME)
wandb.run.name = "[Error-Analysis]"

confusion_matrix = wandb.plot.confusion_matrix(
    probs=None,
    y_true=evals["y_true"],
    preds=evals["y_pred"],
    class_names=[str(i) for i in range(10)]   
)

misclassified_images = wandb.Table(
    columns=["Index", "Image", "Actual", "Predicted"]
)

for i, (y, y_pred) in enumerate(zip(evals["y_true"], evals["y_pred"])):
    if y != y_pred:
        img = np.array(test_images[i]).reshape(28, 28)   # reshape 784 -> 28x28
        misclassified_images.add_data(
            i,
            wandb.Image(img, mode="L"),
            int(y),
            int(y_pred)
        )

run.log({
    "confusion_matrix": confusion_matrix,
    "misclassified_images": misclassified_images
})

run.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [34]:
!python train.py --dataset mnist --epochs 20 --batch_size 64 --learning_rate 0.01 --optimizer adam --hidden_layers 64 64 64 64 --num_neurons 20 --activation sigmoid --loss cross_entropy --weight_init zero --weight_decay 0.0 --wandb_project da6401-assignment1-puneet-id25s027 --question Weight-Initialization-And-Symmetry

Epoch   1/20 : Train Loss = 1.8192, Train Accuracy = 20.729629629629628 | Val Loss = 1.8166, Val Accuracy = 20.2333
Epoch   6/20 : Train Loss = 1.7855, Train Accuracy = 21.25 | Val Loss = 1.7870, Val Accuracy = 20.8500
Epoch  11/20 : Train Loss = 1.7847, Train Accuracy = 21.305555555555557 | Val Loss = 1.7968, Val Accuracy = 20.8500
Epoch  16/20 : Train Loss = 1.7805, Train Accuracy = 20.701851851851853 | Val Loss = 1.7900, Val Accuracy = 20.2667
Epoch  20/20 : Train Loss = 1.7882, Train Accuracy = 20.616666666666667 | Val Loss = 1.8046, Val Accuracy = 20.1500


2026-03-07 16:24:20.897408: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 16:24:27.513484: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\brpun\_netrc.
wandb: Currently logged in as: id25s027 (id25s027-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run x69pcelj
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in c:\My Folder\projects\da6401_assignment_1_id25s027\src\wandb\run-2026030

In [36]:
run = wandb.init(project=PROJECT_NAME)
wandb.run.name = "THe-Fashion-MNIST-Transfer-Challenge"

In [37]:
dataset = "fashion_mnist"

(train_images, train_labels), (test_images, test_labels) = load_dataset(dataset)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 3us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [38]:
train_images, test_images = train_images / 255.0, test_images / 255.0

train_images = train_images.reshape((-1, 28 * 28))
test_images = test_images.reshape((-1, 28 * 28))

output_size = len(np.unique(train_labels).tolist())

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

x_train, x_val, y_train, y_val = train_test_split(train_images, train_labels, test_size = 0.1, random_state = 42)


print(train_images[0].shape[0])

784


In [39]:
def model_run(config, output_activation):
    model = NeuralNetwork(
            cli_args=config,
            input_size = train_images[0].shape[0],
            output_size = 10,
            output_activation = output_activation
        )
    run = wandb.init(
            project = PROJECT_NAME
        )
    wandb.run.name = f"[Fashion-MNIST-TC]-lr={config.learning_rate}_bs={config.batch_size}_opt={config.optimizer}_act={config.activation}_loss={config.loss}_wd={config.weight_decay}_wi={config.weight_init}_sz={config.hidden_layers}_wandbid={wandb.run.id}"
        
    model.train(x_train, y_train, epochs=config.epochs, batch_size=config.batch_size, x_val=x_val, y_val=y_val, wandb=run)

    test_eval = model.evaluate(test_images, test_labels)
        
    test_log = {
            "test_accuracy": test_eval["accuracy"], 
            "test_recall": test_eval["recall"],
            "test_precision": test_eval["precision"],
            "test_f1": test_eval["f1"],
        }


    confusion_matrix = wandb.plot.confusion_matrix(probs=None,
                        y_true=test_eval["y_true"], preds=test_eval["y_pred"],
                        class_names=[0,1,2,3,4,5,6,7,8,9])

    run.log({"confusion_matrix": confusion_matrix, "test_log": test_log })
    run.finish()

    return model

In [40]:

config = {
    "learning_rate": 0.0001,
    "activation": "tanh",
    "epochs": 20,
    "hidden_layers": [64,128,64],
    "loss": "cross_entropy",
    "weight_init": "xavier",
    "weight_decay": 0.0001,
    "batch_size": 16,
    "optimizer": "nag",
    "wandb_project": PROJECT_NAME
}

output_activation = "softmax"

    
cfg = Config(**config)

model = model_run(cfg, output_activation=output_activation)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch   1/20 : Train Loss = 0.4044, Train Accuracy = 85.46851851851852 | Val Loss = 0.4170, Val Accuracy = 84.6500
Epoch   6/20 : Train Loss = 0.3255, Train Accuracy = 88.02222222222223 | Val Loss = 0.3757, Val Accuracy = 86.8667
Epoch  11/20 : Train Loss = 0.2440, Train Accuracy = 90.91111111111111 | Val Loss = 0.3241, Val Accuracy = 88.1333
Epoch  16/20 : Train Loss = 0.2212, Train Accuracy = 91.85740740740741 | Val Loss = 0.3316, Val Accuracy = 88.4167
Epoch  20/20 : Train Loss = 0.1991, Train Accuracy = 92.55925925925926 | Val Loss = 0.3344, Val Accuracy = 88.4000


best_val_loss,▁
convergence_epoch,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
final_train_accuracy,▁
final_train_loss,▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,██▇▇▆▆▆▆▅▅▄▄▄▃▃▂▂▂▂▁
train_accuracy,▁▂▃▄▄▃▅▅▆▆▆▆▇▇▇▇████
train_loss,█▆▆▅▅▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁
+3,...


<Figure size 640x480 with 0 Axes>

In [41]:
model.evaluate(test_images, test_labels)

{'accuracy': 88.22,
 'recall': 0.8822,
 'f1': 0.8822,
 'precision': 0.8822,
 'loss': 0.3433221232609073,
 'logits': array([[-4.60098934, -3.54344921, -3.77894157, ...,  6.01001445,
          0.12682331, 10.16455738],
        [ 4.44445068, -7.36348911, 11.46828063, ..., -6.59654824,
         -4.0499256 , -4.76363991],
        [ 2.20252768, 12.7715205 ,  1.81495925, ..., -2.88589554,
         -0.53553266, -0.55184637],
        ...,
        [ 1.49331143, -2.78435082,  0.13869508, ..., -2.74933273,
         10.98033522, -5.38144121],
        [-0.75415121,  9.71966724,  0.24101993, ..., -2.20458804,
          0.37118848, -0.66930374],
        [-1.82233692, -8.39741906, -1.71200957, ...,  2.80520011,
          0.10697342, -2.28873731]], shape=(10000, 10)),
 'y_true': array([9, 2, 1, ..., 8, 1, 5], shape=(10000,)),
 'y_pred': array([9, 2, 1, ..., 8, 1, 5], shape=(10000,))}

In [42]:

config = {
    "learning_rate": 0.0001,
    "activation": "relu",
    "epochs": 20,
    "hidden_layers": [128,128,128],
    "loss": "cross_entropy",
    "weight_init": "xavier",
    "weight_decay": 0.0,
    "batch_size": 16,
    "optimizer": "adam",
    "wandb_project": PROJECT_NAME
}

output_activation = "softmax"

cfg = Config(**config)

model = model_run(cfg, output_activation=output_activation)

Epoch   1/20 : Train Loss = 0.4338, Train Accuracy = 84.83888888888889 | Val Loss = 0.4411, Val Accuracy = 84.1333
Epoch   6/20 : Train Loss = 0.2916, Train Accuracy = 89.43518518518519 | Val Loss = 0.3325, Val Accuracy = 88.0333
Epoch  11/20 : Train Loss = 0.2401, Train Accuracy = 91.13703703703703 | Val Loss = 0.3186, Val Accuracy = 88.7167
Epoch  16/20 : Train Loss = 0.2158, Train Accuracy = 92.04629629629629 | Val Loss = 0.3233, Val Accuracy = 88.5833
Epoch  20/20 : Train Loss = 0.1836, Train Accuracy = 93.29074074074074 | Val Loss = 0.3208, Val Accuracy = 88.9333


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


best_val_loss,▁
convergence_epoch,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
final_train_accuracy,▁
final_train_loss,▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,█▇▇▇▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▁
train_accuracy,▁▃▃▄▃▅▅▅▅▆▆▆▇▇▇▇█▇██
train_loss,█▆▆▅▅▄▄▃▄▃▃▂▂▂▂▂▁▂▁▁
+3,...


<Figure size 640x480 with 0 Axes>

In [43]:
model.evaluate(test_images, test_labels)

{'accuracy': 88.61,
 'recall': 0.8861,
 'f1': 0.8861,
 'precision': 0.8861,
 'loss': 0.33448353847280154,
 'logits': array([[ -6.86375982, -10.50524934,  -7.13119739, ...,   2.19924725,
          -3.9481333 ,   9.57901799],
        [  1.0129111 , -16.03540904,  11.38552515, ..., -16.47630546,
          -7.76740998, -16.98027322],
        [  0.19194166,  20.40791122,  -4.75728862, ..., -14.54031243,
          -5.79660028, -10.66828541],
        ...,
        [ -1.74612387,  -8.5820597 ,  -1.4090718 , ...,  -8.76545218,
          11.50304584, -12.06291274],
        [ -2.81860984,  11.99643951,  -5.8777664 , ...,  -9.85017196,
          -5.88847498,  -7.01345456],
        [ -6.207914  ,  -7.55578957,  -4.84820655, ...,  -0.21857148,
          -2.53130285,  -7.00654979]], shape=(10000, 10)),
 'y_true': array([9, 2, 1, ..., 8, 1, 5], shape=(10000,)),
 'y_pred': array([9, 2, 1, ..., 8, 1, 5], shape=(10000,))}

In [44]:

config = {
    "learning_rate": 0.001,
    "activation": "relu",
    "epochs": 20,
    "hidden_layers": [32,64,32],
    "loss": "cross_entropy",
    "weight_init": "xavier",
    "weight_decay": 0.0,
    "batch_size": 16,
    "optimizer": "rmsprop",
    "wandb_project": PROJECT_NAME
}

output_activation = "softmax"



    
cfg = Config(**config)

model = model_run(cfg, output_activation=output_activation)

Epoch   1/20 : Train Loss = 0.4124, Train Accuracy = 85.13518518518518 | Val Loss = 0.4261, Val Accuracy = 84.6000
Epoch   6/20 : Train Loss = 0.2978, Train Accuracy = 88.76481481481481 | Val Loss = 0.3600, Val Accuracy = 87.4500
Epoch  11/20 : Train Loss = 0.2914, Train Accuracy = 88.85185185185185 | Val Loss = 0.3718, Val Accuracy = 86.7667
Epoch  16/20 : Train Loss = 0.2697, Train Accuracy = 89.73148148148148 | Val Loss = 0.3778, Val Accuracy = 86.9167
Epoch  20/20 : Train Loss = 0.2345, Train Accuracy = 91.23333333333333 | Val Loss = 0.3671, Val Accuracy = 87.8500


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


best_val_loss,▁
convergence_epoch,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
final_train_accuracy,▁
final_train_loss,▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,█▇▆▆▆▅▅▄▄▄▄▃▃▃▃▂▃▁▁▁
train_accuracy,▁▃▄▄▃▅▆▆▆▇▅▆▆▇▇▆▇▇██
train_loss,█▆▅▄▅▄▃▃▃▂▃▃▃▂▂▃▂▂▁▁
+3,...


<Figure size 640x480 with 0 Axes>

In [45]:
model.evaluate(test_images, test_labels)

{'accuracy': 87.06,
 'recall': 0.8706,
 'f1': 0.8706,
 'precision': 0.8706,
 'loss': 0.3723317656232143,
 'logits': array([[ -9.09141355,  -7.982832  , -17.40637435, ...,  -1.80875449,
         -10.7835869 ,   4.78178489],
        [ -2.58207372, -15.53765045,   6.82874972, ..., -43.67144516,
          -8.04488954, -36.95667064],
        [ -7.90861699,  24.23036114, -15.30453949, ..., -82.43183529,
         -12.79514039, -35.53521118],
        ...,
        [ -0.63830618, -12.42917959,  -3.6836357 , ..., -36.1864464 ,
           8.83236066, -27.66430777],
        [ -4.20800607,   9.6319222 ,  -7.84492289, ..., -17.49473701,
          -5.21015859,  -9.11017632],
        [-10.53509604, -23.35014692, -13.23199647, ...,  -3.79703988,
          -5.5786424 ,  -7.82294995]], shape=(10000, 10)),
 'y_true': array([9, 2, 1, ..., 8, 1, 5], shape=(10000,)),
 'y_pred': array([9, 2, 1, ..., 8, 1, 5], shape=(10000,))}